# Fundamentos de Ingeniería de Datos
## Consultas SQL sobre la base de datos del pipeline (`sqlite3`)

```
SQLite (capa RAW: base_datos.db)  →  SQL  →  resultado (DataFrame)
```

En el notebook de **INGESTA** guardamos los datasets como tablas `raw_*` dentro de `base_datos.db`. Hoy aprendemos a **hacerle preguntas** a esa base de datos usando **SQL** (*Structured Query Language*), el lenguaje estándar para consultar datos en bases relacionales.

**Al terminar este notebook sabrás:**

| # | Tema | Cláusulas / herramientas |
|---|---|---|
| 1 | Abrir el `.db` y explorar qué contiene | `sqlite3.connect`, `sqlite_master`, `PRAGMA table_info` |
| 2 | Seleccionar y filtrar | `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT` |
| 3 | Calcular y clasificar | operadores, `ROUND`, `CAST`, `COALESCE`, `CASE WHEN` |
| 4 | Resumir | `COUNT`, `SUM`, `AVG`, `GROUP BY`, `HAVING` |
| 5 | Combinar consultas y tablas | subconsultas, `WITH` (CTE), `JOIN` |
| 6 | Análisis avanzado | funciones de ventana (`OVER`, `RANK`) |
| 7 | Buenas prácticas | consultas parametrizadas, SQL para revisar calidad |

> **Regla de oro (sigue vigente):** la capa RAW nunca se modifica. Por eso hoy abriremos la base de datos **en modo solo lectura**.

Los ejemplos usan el dataset principal del curso (`raw_df1`, ventas de videojuegos de `gregorut/videogamesales`). Al final aplicarás lo aprendido sobre **tus 3 tablas `raw_<etiqueta>`** de la Actividad 1.

### Configuración del entorno del proyecto

Igual que en INGESTA: montamos Google Drive e importamos `utils.py`.

In [3]:
# ── Montar Google Drive ────────────────────────────────────────────────────

#from google.colab import drive
#drive.mount('/content/drive')

In [4]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys

sys.path.append('/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Utilidades/utils.py')

from utils import PROYECTO_BASE, DB_PATH, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()

ModuleNotFoundError: No module named 'utils'

In [ ]:
# ── Librerías ──────────────────────────────────────────────────────────────

import os
import sqlite3
import pandas as pd

# Mostrar todas las columnas al imprimir DataFrames
pd.set_option('display.max_columns', None)


if os.path.exists(DB_PATH):
    print()
    print(f'Archivo encontrado ({os.path.getsize(DB_PATH) / 1_048_576:.2f} MB)')
    print(f'Versión de SQLite: {sqlite3.sqlite_version}')
    print()
else:
    print('No se encontró base_datos.db: ejecuta primero el notebook de INGESTA.')


Archivo encontrado (1.83 MB)
Versión de SQLite: 3.50.4



---
## PASO 1 — Abrir la base de datos

Recuerda el ciclo de vida de una conexión: **connect → operar → commit → close**.

Hoy solo vamos a **leer**, así que abrimos el archivo en **modo solo lectura** (`mode=ro`). Si por error intentamos modificar una tabla, SQLite lo rechazará: es una protección para la capa RAW.

| Forma de conectar | Qué permite |
|---|---|
| `sqlite3.connect(DB_PATH)` | Leer **y escribir** (la usamos en INGESTA para guardar RAW) |
| `sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)` | **Solo leer**: ideal para consultar sin riesgo |

In [ ]:
# ── Conexión en modo SOLO LECTURA ──────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

print()
print('Conexión abierta en modo solo lectura')
print()


Conexión abierta en modo solo lectura



### Dos maneras de ejecutar una consulta

1. **`sqlite3` puro**: un *cursor* ejecuta la consulta y `fetchall()` regresa una lista de tuplas.
2. **`pandas`**: `pd.read_sql_query()` ejecuta la consulta y regresa un **DataFrame** (más cómodo para ver y analizar).

Las dos usan el mismo SQL; solo cambia el formato del resultado.

In [ ]:
# ── Forma 1: cursor de sqlite3 ─────────────────────────────────────────────

cursor = conn.cursor()
cursor.execute('SELECT Name, Platform, Year FROM raw_df1 LIMIT 3')

filas = cursor.fetchall()            # lista de tuplas

print()
print(type(filas))
for fila in filas:
    print(fila)
print()


<class 'list'>
('Wii Sports', 'Wii', 2006.0)
('Super Mario Bros.', 'NES', 1985.0)
('Mario Kart Wii', 'Wii', 2008.0)



In [ ]:
# ── Forma 2: pandas ────────────────────────────────────────────────────────

df = pd.read_sql_query('SELECT Name, Platform, Year FROM raw_df1 LIMIT 3', conn)

print(type(df))
display(df)

<class 'pandas.DataFrame'>


,Name,Platform,Year
0,Wii Sports,Wii,2006.0
1,Super Mario Bros.,NES,1985.0
2,Mario Kart Wii,Wii,2008.0


### Función auxiliar `consulta()`

Para no repetir `pd.read_sql_query(..., conn)` en cada celda, definimos una función corta. Recibe el texto SQL (y, opcionalmente, parámetros; los veremos en el Paso 9) y regresa un DataFrame.

In [ ]:
def consulta(sql, params=None):
    '''Ejecuta una consulta SQL sobre la conexión abierta y regresa un DataFrame.'''
    return pd.read_sql_query(sql, conn, params=params)


print()
print('Función consulta() lista')
print()


Función consulta() lista



---
## PASO 2 — Explorar la base de datos antes de consultar

Antes de escribir consultas hay que saber **qué tablas existen** y **qué columnas y tipos** tienen.

- `sqlite_master` es la tabla especial donde SQLite guarda el catálogo de todo lo que contiene el archivo (es lo que usa `listar_tablas()` de `utils.py`).
- `PRAGMA table_info(tabla)` describe las columnas de una tabla: nombre, tipo, si acepta nulos y si es clave primaria.

In [ ]:
# ── ¿Qué tablas hay en el archivo? ─────────────────────────────────────────

consulta('''
    SELECT name AS tabla, type AS tipo
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
''')

,tabla,tipo
0,raw_df1,table


In [ ]:
# ── ¿Qué columnas tiene raw_df1 y de qué tipo son? ─────────────────────────

consulta('PRAGMA table_info(raw_df1)')

,cid,name,type,notnull,dflt_value,pk
0,0,Rank,INTEGER,0,None,0
1,1,Name,TEXT,0,None,0
2,2,Platform,TEXT,0,None,0
3,3,Year,REAL,0,None,0
4,4,Genre,TEXT,0,None,0
5,5,Publisher,TEXT,0,None,0
6,6,NA_Sales,REAL,0,None,0
7,7,EU_Sales,REAL,0,None,0
8,8,JP_Sales,REAL,0,None,0
9,9,Other_Sales,REAL,0,None,0


**Lectura del resultado:** `type` es el tipo con el que `pandas` creó cada columna al hacer `to_sql()` en INGESTA:

| Tipo pandas | Tipo SQLite |
|---|---|
| `int64` | `INTEGER` |
| `float64` | `REAL` |
| `object` (texto) | `TEXT` |

Observa que `Year` quedó como `REAL` (decimal) y no como entero: en el CSV hay años faltantes, y pandas representa los faltantes como `NaN`, que solo existe en columnas decimales. En SQLite esos faltantes se guardan como `NULL`.

In [ ]:
# ── ¿Cuántos registros tiene? ──────────────────────────────────────────────

consulta('SELECT COUNT(*) AS total_registros FROM raw_df1')

,total_registros
0,16598


---
## PASO 3 — `SELECT`: elegir columnas

```sql
SELECT columna1, columna2      -- ¿qué columnas quiero?
FROM   tabla                   -- ¿de qué tabla?
LIMIT  n                       -- ¿cuántas filas como máximo?
```

- `*` significa "todas las columnas".
- `AS` le pone un **alias** (nombre nuevo) a una columna del resultado.
- `DISTINCT` elimina filas repetidas del resultado.
- SQL **no distingue mayúsculas** en las palabras clave (`select` = `SELECT`), pero por convención se escriben en mayúsculas.

In [ ]:
# ── Todas las columnas, primeras 5 filas ───────────────────────────────────

consulta('SELECT * FROM raw_df1 LIMIT 5')

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [ ]:
# ── Solo algunas columnas, con alias ───────────────────────────────────────

consulta('''
    SELECT Name         AS juego,
           Platform     AS plataforma,
           Global_Sales AS ventas_millones
    FROM raw_df1
    LIMIT 5
''')

,juego,plataforma,ventas_millones
0,Wii Sports,Wii,82.74
1,Super Mario Bros.,NES,40.24
2,Mario Kart Wii,Wii,35.82
3,Wii Sports Resort,Wii,33.00
4,Pokemon Red/Pokemon Blue,GB,31.37


In [ ]:
# ── DISTINCT: valores únicos ───────────────────────────────────────────────

consulta('SELECT DISTINCT Genre FROM raw_df1')

,Genre
0,Sports
1,Platform
2,Racing
3,Role-Playing
4,Puzzle
5,Misc
6,Shooter
7,Simulation
8,Action
9,Fighting


In [ ]:
# ── ¿Cuántos valores distintos hay en cada columna categórica? ─────────────

consulta('''
    SELECT COUNT(DISTINCT Platform)  AS plataformas,
           COUNT(DISTINCT Genre)     AS generos,
           COUNT(DISTINCT Publisher) AS editoras
    FROM raw_df1
''')

,plataformas,generos,editoras
0,31,12,578


---
## PASO 4 — `WHERE`: filtrar filas

`WHERE` se queda solo con las filas que cumplen una condición.

| Operador | Ejemplo | Significado |
|---|---|---|
| `=`, `<>`, `>`, `<`, `>=`, `<=` | `Year >= 2010` | comparación |
| `AND`, `OR`, `NOT` | `Genre = 'Sports' AND Year = 2008` | combinar condiciones |
| `IN (...)` | `Platform IN ('PS4', 'XOne')` | pertenece a una lista |
| `BETWEEN a AND b` | `Year BETWEEN 2000 AND 2009` | rango **inclusivo** |
| `LIKE` | `Name LIKE '%Mario%'` | patrón de texto (`%` = cualquier texto, `_` = un carácter) |
| `IS NULL` / `IS NOT NULL` | `Publisher IS NULL` | valor faltante |

> Los textos van entre **comillas simples** `'...'`.

In [ ]:
# ── Comparación simple ─────────────────────────────────────────────────────

consulta('''
    SELECT Name, Platform, Year, Global_Sales
    FROM raw_df1
    WHERE Global_Sales > 20
''')

,Name,Platform,Year,Global_Sales
0,Wii Sports,Wii,2006.0,82.74
1,Super Mario Bros.,NES,1985.0,40.24
2,Mario Kart Wii,Wii,2008.0,35.82
3,Wii Sports Resort,Wii,2009.0,33.00
4,Pokemon Red/Pokemon Blue,GB,1996.0,31.37
5,Tetris,GB,1989.0,30.26
6,New Super Mario Bros.,DS,2006.0,30.01
7,Wii Play,Wii,2006.0,29.02
8,New Super Mario Bros. Wii,Wii,2009.0,28.62
9,Duck Hunt,NES,1984.0,28.31


In [ ]:
# ── AND / OR: combinar condiciones ─────────────────────────────────────────

consulta('''
    SELECT Name, Platform, Year, Genre, Global_Sales
    FROM raw_df1
    WHERE Genre = 'Sports'
      AND (Platform = 'PS2' OR Platform = 'Wii')
    LIMIT 10
''')

,Name,Platform,Year,Genre,Global_Sales
0,Wii Sports,Wii,2006.0,Sports,82.74
1,Wii Sports Resort,Wii,2009.0,Sports,33.00
2,Wii Fit,Wii,2007.0,Sports,22.72
3,Wii Fit Plus,Wii,2009.0,Sports,22.00
4,Mario & Sonic at the Olympic Games,Wii,2007.0,Sports,8.06
5,Zumba Fitness,Wii,2010.0,Sports,6.81
6,Madden NFL 2004,PS2,NaN,Sports,5.23
7,Madden NFL 06,PS2,2005.0,Sports,4.91
8,Mario & Sonic at the Olympic Winter Games,Wii,2009.0,Sports,4.58
9,Madden NFL 2005,PS2,2004.0,Sports,4.53


In [ ]:
# ── IN y BETWEEN ───────────────────────────────────────────────────────────

consulta('''
    SELECT Name, Platform, Year, Global_Sales
    FROM raw_df1
    WHERE Platform IN ('PS4', 'XOne', 'WiiU')
      AND Year BETWEEN 2014 AND 2015
    LIMIT 10
''')

,Name,Platform,Year,Global_Sales
0,Call of Duty: Black Ops 3,PS4,2015.0,14.24
1,Grand Theft Auto V,PS4,2014.0,11.98
2,FIFA 16,PS4,2015.0,8.49
3,Star Wars Battlefront (2015),PS4,2015.0,7.67
4,Call of Duty: Advanced Warfare,PS4,2014.0,7.60
5,Call of Duty: Black Ops 3,XOne,2015.0,7.30
6,Mario Kart 8,WiiU,2014.0,6.96
7,Fallout 4,PS4,2015.0,6.96
8,FIFA 15,PS4,2014.0,6.59
9,Destiny,PS4,2014.0,5.65


In [ ]:
# ── LIKE: buscar por patrón de texto ───────────────────────────────────────

consulta('''
    SELECT Name, Platform, Year
    FROM raw_df1
    WHERE Name LIKE '%Mario%'
    LIMIT 10
''')

,Name,Platform,Year
0,Super Mario Bros.,NES,1985.0
1,Mario Kart Wii,Wii,2008.0
2,New Super Mario Bros.,DS,2006.0
3,New Super Mario Bros. Wii,Wii,2009.0
4,Mario Kart DS,DS,2005.0
5,Super Mario World,SNES,1990.0
6,Super Mario Land,GB,1989.0
7,Super Mario Bros. 3,NES,1988.0
8,Mario Kart 7,3DS,2011.0
9,Super Mario 64,N64,1996.0


### Cuidado con `NULL`

`NULL` significa **"valor desconocido"**, no cero ni texto vacío. Por eso **no se compara con `=`**: `Year = NULL` nunca es verdadero. Siempre se usa `IS NULL` o `IS NOT NULL`.

In [ ]:
# ── IS NULL vs = NULL ──────────────────────────────────────────────────────

consulta('''
    SELECT
        (SELECT COUNT(*) FROM raw_df1 WHERE Year = NULL)  AS con_igual,
        (SELECT COUNT(*) FROM raw_df1 WHERE Year IS NULL) AS con_is_null
''')

,con_igual,con_is_null
0,0,271


---
## PASO 5 — `ORDER BY` y `LIMIT`: ordenar y quedarse con los primeros

- `ORDER BY columna ASC` ordena de menor a mayor (es el valor por defecto).
- `ORDER BY columna DESC` ordena de mayor a menor.
- Se puede ordenar por varias columnas: primero por la primera y, en caso de empate, por la segunda.
- `ORDER BY ... DESC LIMIT n` es la receta para obtener un **"Top n"**.

In [ ]:
# ── Top 10 de juegos más vendidos ──────────────────────────────────────────

consulta('''
    SELECT Name, Platform, Year, Publisher, Global_Sales
    FROM raw_df1
    ORDER BY Global_Sales DESC
    LIMIT 10
''')

,Name,Platform,Year,Publisher,Global_Sales
0,Wii Sports,Wii,2006.0,Nintendo,82.74
1,Super Mario Bros.,NES,1985.0,Nintendo,40.24
2,Mario Kart Wii,Wii,2008.0,Nintendo,35.82
3,Wii Sports Resort,Wii,2009.0,Nintendo,33.00
4,Pokemon Red/Pokemon Blue,GB,1996.0,Nintendo,31.37
5,Tetris,GB,1989.0,Nintendo,30.26
6,New Super Mario Bros.,DS,2006.0,Nintendo,30.01
7,Wii Play,Wii,2006.0,Nintendo,29.02
8,New Super Mario Bros. Wii,Wii,2009.0,Nintendo,28.62
9,Duck Hunt,NES,1984.0,Nintendo,28.31


In [ ]:
# ── Ordenar por dos columnas ───────────────────────────────────────────────

consulta('''
    SELECT Platform, Name, Global_Sales
    FROM raw_df1
    WHERE Platform IN ('GB', 'NES')
    ORDER BY Platform ASC, Global_Sales DESC
    LIMIT 10
''')

,Platform,Name,Global_Sales
0,GB,Pokemon Red/Pokemon Blue,31.37
1,GB,Tetris,30.26
2,GB,Pokemon Gold/Pokemon Silver,23.10
3,GB,Super Mario Land,18.14
4,GB,Pokémon Yellow: Special Pikachu Edition,14.64
5,GB,Super Mario Land 2: 6 Golden Coins,11.18
6,GB,Pokémon Crystal Version,6.39
7,GB,Dr. Mario,5.34
8,GB,Pokemon Pinball,5.31
9,GB,Super Mario Land 3: Wario Land,5.19


---
## PASO 6 — Columnas calculadas y funciones

En el `SELECT` no solo se eligen columnas: también se pueden **calcular** columnas nuevas. Esto no modifica la tabla; el cálculo solo existe en el resultado.

| Función | Para qué sirve | Ejemplo |
|---|---|---|
| `+ - * /` | aritmética | `NA_Sales + EU_Sales` |
| `ROUND(x, n)` | redondear a `n` decimales | `ROUND(Global_Sales, 1)` |
| `CAST(x AS tipo)` | convertir tipo | `CAST(Year AS INTEGER)` |
| `UPPER()`, `LOWER()`, `LENGTH()`, `SUBSTR()` | funciones de texto | `UPPER(Platform)` |
| `COALESCE(x, valor)` | reemplazar `NULL` por un valor | `COALESCE(Publisher, 'Desconocida')` |
| `CASE WHEN ... THEN ... ELSE ... END` | clasificar con reglas (como un `if`) | ver abajo |

In [ ]:
# ── Aritmética, ROUND y CAST ───────────────────────────────────────────────

consulta('''
    SELECT Name,
           CAST(Year AS INTEGER)                          AS anio,
           Global_Sales,
           ROUND(NA_Sales / Global_Sales * 100, 1)        AS pct_norteamerica,
           ROUND(JP_Sales / Global_Sales * 100, 1)        AS pct_japon
    FROM raw_df1
    ORDER BY Global_Sales DESC
    LIMIT 10
''')

,Name,anio,Global_Sales,pct_norteamerica,pct_japon
0,Wii Sports,2006,82.74,50.1,4.6
1,Super Mario Bros.,1985,40.24,72.3,16.9
2,Mario Kart Wii,2008,35.82,44.2,10.6
3,Wii Sports Resort,2009,33.00,47.7,9.9
4,Pokemon Red/Pokemon Blue,1996,31.37,35.9,32.6
5,Tetris,1989,30.26,76.7,13.9
6,New Super Mario Bros.,2006,30.01,37.9,21.7
7,Wii Play,2006,29.02,48.3,10.1
8,New Super Mario Bros. Wii,2009,28.62,51.0,16.4
9,Duck Hunt,1984,28.31,95.1,1.0


In [ ]:
# ── COALESCE y funciones de texto ──────────────────────────────────────────

consulta('''
    SELECT Name,
           LENGTH(Name)                        AS largo_nombre,
           UPPER(Platform)                     AS plataforma,
           COALESCE(Publisher, 'Desconocida')  AS editora
    FROM raw_df1
    WHERE Publisher IS NULL
    LIMIT 5
''')

,Name,largo_nombre,plataforma,editora
0,wwe Smackdown vs. Raw 2006,26,PS2,Desconocida
1,Triple Play 99,14,PS,Desconocida
2,Shrek / Shrek 2 2-in-1 Gameboy Advance Video,44,GBA,Desconocida
3,Bentley's Hackpack,18,GBA,Desconocida
4,Nicktoons Collection: Game Boy Advance Video V...,53,GBA,Desconocida


In [ ]:
# ── CASE WHEN: clasificar con reglas ───────────────────────────────────────

consulta('''
    SELECT Name, Global_Sales,
           CASE
               WHEN Global_Sales >= 10 THEN 'Blockbuster'
               WHEN Global_Sales >= 1  THEN 'Éxito'
               ELSE                         'Nicho'
           END AS categoria
    FROM raw_df1
    ORDER BY Global_Sales DESC
    LIMIT 10
''')

,Name,Global_Sales,categoria
0,Wii Sports,82.74,Blockbuster
1,Super Mario Bros.,40.24,Blockbuster
2,Mario Kart Wii,35.82,Blockbuster
3,Wii Sports Resort,33.00,Blockbuster
4,Pokemon Red/Pokemon Blue,31.37,Blockbuster
5,Tetris,30.26,Blockbuster
6,New Super Mario Bros.,30.01,Blockbuster
7,Wii Play,29.02,Blockbuster
8,New Super Mario Bros. Wii,28.62,Blockbuster
9,Duck Hunt,28.31,Blockbuster


---
## PASO 7 — Agregaciones: `GROUP BY` y `HAVING`

Las **funciones de agregación** resumen muchas filas en un solo valor:

| Función | Resultado |
|---|---|
| `COUNT(*)` | número de filas |
| `COUNT(columna)` | número de valores **no nulos** |
| `SUM()`, `AVG()` | suma, promedio |
| `MIN()`, `MAX()` | mínimo, máximo |

- Con **`GROUP BY`**, la agregación se calcula **por grupo** (una fila del resultado por cada valor distinto).
- **`WHERE`** filtra filas **antes** de agrupar; **`HAVING`** filtra grupos **después** de agrupar.

In [ ]:
# ── Agregación sobre toda la tabla ─────────────────────────────────────────

consulta('''
    SELECT COUNT(*)                     AS juegos,
           COUNT(Year)                  AS juegos_con_anio,
           ROUND(SUM(Global_Sales), 2)  AS ventas_totales,
           ROUND(AVG(Global_Sales), 3)  AS ventas_promedio,
           MIN(Year)                    AS primer_anio,
           MAX(Year)                    AS ultimo_anio
    FROM raw_df1
''')

,juegos,juegos_con_anio,ventas_totales,ventas_promedio,primer_anio,ultimo_anio
0,16598,16327,8920.44,0.537,1980.0,2020.0


In [ ]:
# ── GROUP BY: ventas por género ────────────────────────────────────────────

consulta('''
    SELECT Genre                        AS genero,
           COUNT(*)                     AS juegos,
           ROUND(SUM(Global_Sales), 2)  AS ventas_totales,
           ROUND(AVG(Global_Sales), 3)  AS ventas_promedio
    FROM raw_df1
    GROUP BY Genre
    ORDER BY ventas_totales DESC
''')

,genero,juegos,ventas_totales,ventas_promedio
0,Action,3316,1751.18,0.528
1,Sports,2346,1330.93,0.567
2,Shooter,1310,1037.37,0.792
3,Role-Playing,1488,927.37,0.623
4,Platform,886,831.37,0.938
5,Misc,1739,809.96,0.466
6,Racing,1249,732.04,0.586
7,Fighting,848,448.91,0.529
8,Simulation,867,392.20,0.452
9,Puzzle,582,244.95,0.421


In [ ]:
# ── GROUP BY con dos columnas: ventas por plataforma y género ──────────────

consulta('''
    SELECT Platform, Genre,
           ROUND(SUM(Global_Sales), 2) AS ventas
    FROM raw_df1
    WHERE Platform IN ('PS4', 'XOne')
    GROUP BY Platform, Genre
    ORDER BY Platform, ventas DESC
    LIMIT 12
''')

,Platform,Genre,ventas
0,PS4,Action,87.06
1,PS4,Shooter,75.32
2,PS4,Sports,50.07
3,PS4,Role-Playing,25.77
4,PS4,Racing,11.53
5,PS4,Fighting,8.04
6,PS4,Misc,7.40
7,PS4,Platform,7.01
8,PS4,Adventure,4.70
9,PS4,Simulation,0.77


In [ ]:
# ── HAVING: editoras con al menos 100 juegos ───────────────────────────────

consulta('''
    SELECT Publisher,
           COUNT(*)                     AS juegos,
           ROUND(SUM(Global_Sales), 2)  AS ventas_totales
    FROM raw_df1
    WHERE Publisher IS NOT NULL          -- filtra FILAS (antes de agrupar)
    GROUP BY Publisher
    HAVING COUNT(*) >= 100               -- filtra GRUPOS (después de agrupar)
    ORDER BY ventas_totales DESC
''')

,Publisher,juegos,ventas_totales
0,Nintendo,703,1786.56
1,Electronic Arts,1351,1110.32
2,Activision,975,727.46
3,Sony Computer Entertainment,683,607.50
4,Ubisoft,921,474.72
5,Take-Two Interactive,413,399.54
6,THQ,715,340.77
7,Konami Digital Entertainment,832,283.64
8,Sega,639,272.99
9,Namco Bandai Games,932,254.09


### Orden en que se **escribe** vs. orden en que se **ejecuta**

Escribimos la consulta en un orden, pero el motor la procesa en otro. Entender esto explica por qué un alias del `SELECT` no puede usarse en el `WHERE`, pero sí en el `ORDER BY`.

| Orden de escritura | Orden de ejecución |
|---|---|
| 1. `SELECT` | 1. `FROM` — de qué tabla |
| 2. `FROM` | 2. `WHERE` — filtra filas |
| 3. `WHERE` | 3. `GROUP BY` — forma grupos |
| 4. `GROUP BY` | 4. `HAVING` — filtra grupos |
| 5. `HAVING` | 5. `SELECT` — calcula columnas y alias |
| 6. `ORDER BY` | 6. `ORDER BY` — ordena |
| 7. `LIMIT` | 7. `LIMIT` — recorta |

---
## PASO 8 — Combinar consultas: subconsultas, `WITH` y `JOIN`

### 8.1 Subconsultas

Una **subconsulta** es una consulta dentro de otra, escrita entre paréntesis. Su resultado se usa como valor o como tabla.

In [ ]:
# ── Juegos de 2010 que vendieron más que el promedio de ese año ────────────

consulta('''
    SELECT Name, Platform, Global_Sales
    FROM raw_df1
    WHERE Year = 2010
      AND Global_Sales > (SELECT AVG(Global_Sales)
                          FROM raw_df1
                          WHERE Year = 2010)
    ORDER BY Global_Sales DESC
    LIMIT 10
''')

,Name,Platform,Global_Sales
0,Kinect Adventures!,X360,21.82
1,Pokemon Black/Pokemon White,DS,15.32
2,Call of Duty: Black Ops,X360,14.64
3,Call of Duty: Black Ops,PS3,12.73
4,Gran Turismo 5,PS3,10.77
5,Halo: Reach,X360,9.88
6,Just Dance 2,Wii,9.52
7,Wii Party,Wii,8.49
8,Super Mario Galaxy 2,Wii,7.69
9,Zumba Fitness,Wii,6.81


### 8.2 `WITH` (CTE: *Common Table Expression*)

`WITH` le pone **nombre a un resultado intermedio** para usarlo después, como si fuera una tabla. Hace que las consultas largas se lean de arriba hacia abajo, paso a paso, igual que un pipeline.

In [ ]:
# ── Ventas por década en dos pasos ─────────────────────────────────────────

consulta('''
    WITH ventas_por_anio AS (                     -- paso 1: resumir por año
        SELECT CAST(Year AS INTEGER) AS anio,
               SUM(Global_Sales)     AS ventas
        FROM raw_df1
        WHERE Year IS NOT NULL
        GROUP BY Year
    )
    SELECT (anio / 10) * 10           AS decada,  -- paso 2: agrupar años en décadas
           COUNT(*)                   AS anios_con_datos,
           ROUND(SUM(ventas), 2)      AS ventas_totales
    FROM ventas_por_anio
    GROUP BY (anio / 10) * 10
    ORDER BY decada
''')

,decada,anios_con_datos,ventas_totales
0,1980,10,376.58
1,1990,10,1278.91
2,2000,10,4644.02
3,2010,8,2520.56
4,2020,1,0.29


### 8.3 `JOIN`: unir tablas por una clave

Un `JOIN` combina filas de dos tablas que comparten un valor (la **clave**). Es la base para integrar fuentes distintas, como tus 3 datasets.

Como `raw_df1` es una sola tabla, creamos un pequeño catálogo de plataformas como **tabla temporal** (`TEMP`): vive solo mientras la conexión está abierta y **no se guarda en `base_datos.db`**, así que la capa RAW no cambia.

| Tipo | Qué devuelve |
|---|---|
| `INNER JOIN` | solo las filas que tienen pareja en **ambas** tablas |
| `LEFT JOIN` | **todas** las filas de la tabla izquierda; si no hay pareja, las columnas de la derecha quedan en `NULL` |

In [ ]:
# ── Crear un catálogo temporal de plataformas ──────────────────────────────

conn.execute('DROP TABLE IF EXISTS temp.catalogo_plataformas')
conn.execute('''
    CREATE TEMP TABLE catalogo_plataformas (
        Platform   TEXT PRIMARY KEY,
        fabricante TEXT,
        tipo       TEXT
    )
''')

catalogo = [
    ('Wii',  'Nintendo',  'Sobremesa'), ('DS',   'Nintendo',  'Portátil'),
    ('GB',   'Nintendo',  'Portátil'),  ('NES',  'Nintendo',  'Sobremesa'),
    ('3DS',  'Nintendo',  'Portátil'),  ('WiiU', 'Nintendo',  'Sobremesa'),
    ('PS',   'Sony',      'Sobremesa'), ('PS2',  'Sony',      'Sobremesa'),
    ('PS3',  'Sony',      'Sobremesa'), ('PS4',  'Sony',      'Sobremesa'),
    ('PSP',  'Sony',      'Portátil'),  ('X360', 'Microsoft', 'Sobremesa'),
    ('XOne', 'Microsoft', 'Sobremesa'), ('PC',   'Varios',    'Computadora'),
]
conn.executemany('INSERT INTO catalogo_plataformas VALUES (?, ?, ?)', catalogo)

print()
print(f'Catálogo temporal creado con {len(catalogo)} plataformas (no se guarda en el .db)')
print()


Catálogo temporal creado con 14 plataformas (no se guarda en el .db)



In [ ]:
# ── INNER JOIN: ventas por fabricante ──────────────────────────────────────

consulta('''
    SELECT c.fabricante,
           COUNT(*)                       AS juegos,
           ROUND(SUM(v.Global_Sales), 2)  AS ventas_totales
    FROM raw_df1 AS v
    INNER JOIN catalogo_plataformas AS c
            ON v.Platform = c.Platform
    GROUP BY c.fabricante
    ORDER BY ventas_totales DESC
''')

,fabricante,juegos,ventas_totales
0,Sony,6235,3518.52
1,Nintendo,4336,2585.04
2,Microsoft,1478,1121.02
3,Varios,960,258.82


In [ ]:
# ── LEFT JOIN: ¿qué plataformas NO están en el catálogo? ───────────────────

consulta('''
    SELECT v.Platform,
           COUNT(*) AS juegos
    FROM raw_df1 AS v
    LEFT JOIN catalogo_plataformas AS c
           ON v.Platform = c.Platform
    WHERE c.Platform IS NULL          -- sin pareja en el catálogo
    GROUP BY v.Platform
    ORDER BY juegos DESC
''')

,Platform,juegos
0,XB,824
1,GBA,822
2,GC,556
3,PSV,413
4,N64,319
5,SNES,239
6,SAT,173
7,2600,133
8,DC,52
9,GEN,27


> **Lectura:** el `LEFT JOIN` + `WHERE ... IS NULL` es una técnica clásica para encontrar **registros huérfanos** (sin pareja). En un pipeline real sirve para detectar claves que no existen en un catálogo: un problema de **consistencia** entre fuentes.

> Los **alias de tabla** (`v`, `c`) son abreviaturas para no escribir el nombre completo; `v.Platform` significa "la columna Platform de la tabla v".

---
## PASO 9 — Funciones de ventana (`OVER`)

Una función de ventana calcula un valor **sobre un grupo de filas relacionadas, sin colapsarlas** (a diferencia de `GROUP BY`, que deja una fila por grupo).

```sql
RANK() OVER (PARTITION BY Genre ORDER BY Global_Sales DESC)
--           └─ reinicia por grupo   └─ criterio de orden dentro del grupo
```

In [ ]:
# ── Top 3 juegos de cada género ────────────────────────────────────────────

consulta('''
    WITH ranking AS (
        SELECT Genre, Name, Platform, Global_Sales,
               RANK() OVER (PARTITION BY Genre
                            ORDER BY Global_Sales DESC) AS posicion
        FROM raw_df1
    )
    SELECT *
    FROM ranking
    WHERE posicion <= 3
    ORDER BY Genre, posicion
''')

,Genre,Name,Platform,Global_Sales,posicion
0,Action,Grand Theft Auto V,PS3,21.40,1
1,Action,Grand Theft Auto: San Andreas,PS2,20.81,2
2,Action,Grand Theft Auto V,X360,16.38,3
3,Adventure,Super Mario Land 2: 6 Golden Coins,GB,11.18,1
4,Adventure,Assassin's Creed,X360,5.55,2
5,Adventure,Assassin's Creed,PS3,4.83,3
6,Fighting,Super Smash Bros. Brawl,Wii,13.04,1
7,Fighting,Super Smash Bros. for Wii U and 3DS,3DS,7.45,2
8,Fighting,Tekken 3,PS,7.16,3
9,Misc,Wii Play,Wii,29.02,1


In [ ]:
# ── Acumulado de ventas año con año ────────────────────────────────────────

consulta('''
    WITH por_anio AS (
        SELECT CAST(Year AS INTEGER) AS anio,
               SUM(Global_Sales)     AS ventas
        FROM raw_df1
        WHERE Year IS NOT NULL
        GROUP BY Year
    )
    SELECT anio,
           ROUND(ventas, 2)                             AS ventas_del_anio,
           ROUND(SUM(ventas) OVER (ORDER BY anio), 2)   AS ventas_acumuladas
    FROM por_anio
    ORDER BY anio
''')

,anio,ventas_del_anio,ventas_acumuladas
0,1980,11.38,11.38
1,1981,35.77,47.15
2,1982,28.86,76.01
3,1983,16.79,92.80
4,1984,50.36,143.16
5,1985,53.94,197.10
6,1986,37.07,234.17
7,1987,21.74,255.91
8,1988,47.22,303.13
9,1989,73.45,376.58


---
## PASO 10 — Consultas parametrizadas (buena práctica)

Cuando un valor de la consulta viene de una variable (por ejemplo, lo que escribe un usuario), **no se pega con f-strings**. Se usa un marcador **`?`** y el valor se pasa aparte con `params`.

**¿Por qué?** Si el texto se pega directamente, alguien puede escribir un valor que **cambie el sentido de la consulta** (esto se llama *SQL injection*). Con `?`, SQLite trata el valor siempre como un **dato**, nunca como código.

In [ ]:
# ── Forma INSEGURA (solo para ver el problema) ─────────────────────────────

genero = "Puzzle' OR '1'='1"          # valor "malicioso" escrito por un usuario

sql_inseguro = f"SELECT COUNT(*) AS filas FROM raw_df1 WHERE Genre = '{genero}'"
print()
print('SQL que realmente se ejecuta:')
print(sql_inseguro)
print()
consulta(sql_inseguro)                # ¡regresa TODA la tabla, no solo Puzzle!


SQL que realmente se ejecuta:
SELECT COUNT(*) AS filas FROM raw_df1 WHERE Genre = 'Puzzle' OR '1'='1'



,filas
0,16598


In [ ]:
# ── Forma SEGURA: marcador ? + params ──────────────────────────────────────

consulta('SELECT COUNT(*) AS filas FROM raw_df1 WHERE Genre = ?', params=(genero,))

,filas
0,0


In [ ]:
# ── Uso normal: consultas reutilizables con parámetros ─────────────────────

def top_por_plataforma(plataforma, n=5):
    return consulta('''
        SELECT Name, Year, Global_Sales
        FROM raw_df1
        WHERE Platform = ?
        ORDER BY Global_Sales DESC
        LIMIT ?
    ''', params=(plataforma, n))


display(top_por_plataforma('PS4'))
display(top_por_plataforma('GB', n=3))

,Name,Year,Global_Sales
0,Call of Duty: Black Ops 3,2015.0,14.24
1,Grand Theft Auto V,2014.0,11.98
2,FIFA 16,2015.0,8.49
3,Star Wars Battlefront (2015),2015.0,7.67
4,Call of Duty: Advanced Warfare,2014.0,7.60


,Name,Year,Global_Sales
0,Pokemon Red/Pokemon Blue,1996.0,31.37
1,Tetris,1989.0,30.26
2,Pokemon Gold/Pokemon Silver,1999.0,23.10


---
## PASO 11 — SQL para revisar la calidad de los datos

En ingeniería de datos, SQL no solo responde preguntas de negocio: también sirve para **verificar** que los datos cumplen las reglas que escribiste en tu **miniespecificación** (Actividad 1). Estas consultas son la base de las validaciones que automatizaremos en la Unidad 2.

In [ ]:
# ── Completitud: nulos por columna ─────────────────────────────────────────

consulta('''
    SELECT COUNT(*)                                     AS total,
           SUM(CASE WHEN Name      IS NULL THEN 1 ELSE 0 END) AS nulos_name,
           SUM(CASE WHEN Year      IS NULL THEN 1 ELSE 0 END) AS nulos_year,
           SUM(CASE WHEN Publisher IS NULL THEN 1 ELSE 0 END) AS nulos_publisher
    FROM raw_df1
''')

,total,nulos_name,nulos_year,nulos_publisher
0,16598,0,271,58


In [ ]:
# ── Unicidad: ¿hay registros duplicados (mismo juego, plataforma y año)? ───

consulta('''
    SELECT Name, Platform, Year, COUNT(*) AS veces
    FROM raw_df1
    GROUP BY Name, Platform, Year
    HAVING COUNT(*) > 1
    ORDER BY veces DESC
''')

,Name,Platform,Year,veces
0,Madden NFL 13,PS3,2012.0,2
1,Wii de Asobu: Metroid Prime,Wii,NaN,2


In [ ]:
# ── Rango válido: años fuera de lo esperado ────────────────────────────────

consulta('''
    SELECT CAST(Year AS INTEGER) AS anio, COUNT(*) AS juegos
    FROM raw_df1
    WHERE Year < 1980 OR Year > 2016
    GROUP BY Year
''')

,anio,juegos
0,2017,3
1,2020,1


In [ ]:
# ── Consistencia: ¿la suma de regiones coincide con Global_Sales? ──────────

consulta('''
    SELECT Name, Platform,
           ROUND(NA_Sales + EU_Sales + JP_Sales + Other_Sales, 2) AS suma_regiones,
           Global_Sales,
           ROUND(Global_Sales - (NA_Sales + EU_Sales + JP_Sales + Other_Sales), 2) AS diferencia
    FROM raw_df1
    WHERE ABS(Global_Sales - (NA_Sales + EU_Sales + JP_Sales + Other_Sales)) > 0.02
    ORDER BY ABS(diferencia) DESC
    LIMIT 10
''')

,Name,Platform,suma_regiones,Global_Sales,diferencia
0,Tekken 3,PS,7.18,7.16,-0.02
1,Need For Speed: Undercover,PS2,1.42,1.40,-0.02
2,Need for Speed: The Run,X360,1.36,1.38,0.02
3,Kid Icarus: Uprising,3DS,1.26,1.28,0.02
4,Wet,PS3,0.61,0.63,0.02
5,Yu-Gi-Oh! GX: Tag Force 2,PSP,0.44,0.42,-0.02
6,One Piece: Unlimited World Red,PS3,0.32,0.30,-0.02


> **Pregunta para discutir:** si encuentras duplicados o años fuera de rango, ¿los corriges en RAW? **No**: RAW se queda como llegó. Las correcciones se hacen en las capas siguientes (EXTRACTED / TRANSFORMED) y la regla se documenta.

---
## PASO 12 — ¿Y si quiero crear o modificar tablas? (zona de práctica)

SQL también tiene comandos para **definir** y **modificar** datos:

| Tipo | Comandos | Para qué |
|---|---|---|
| **DDL** (*Data Definition Language*) | `CREATE TABLE`, `DROP TABLE`, `ALTER TABLE` | crear o cambiar la estructura |
| **DML** (*Data Manipulation Language*) | `INSERT`, `UPDATE`, `DELETE` | agregar, cambiar o borrar filas |
| **DQL** (*Data Query Language*) | `SELECT` | consultar (todo lo que hicimos hasta ahora) |

Primero comprobamos que nuestra conexión de solo lectura **protege** la capa RAW:

In [ ]:
# ── Intentar modificar RAW con la conexión de solo lectura ─────────────────

try:
    conn.execute("DELETE FROM raw_df1 WHERE Genre = 'Puzzle'")
except sqlite3.OperationalError as e:
    print()
    print(f'SQLite rechazó la operación: {e}')
    print('La capa RAW sigue intacta.')
    print()


SQLite rechazó la operación: attempt to write a readonly database
La capa RAW sigue intacta.



Para practicar DDL y DML usamos una **base de datos en memoria** (`:memory:`): existe solo mientras la conexión está abierta y desaparece al cerrarla. Así se practica sin riesgo.

In [ ]:
# ── Base de práctica en memoria ────────────────────────────────────────────

sandbox = sqlite3.connect(':memory:')

# DDL: crear una tabla con clave primaria y restricciones
sandbox.execute('''
    CREATE TABLE alumnos (
        id        INTEGER PRIMARY KEY,
        nombre    TEXT    NOT NULL,
        carrera   TEXT,
        promedio  REAL    CHECK (promedio BETWEEN 0 AND 100)
    )
''')

# DML: insertar filas (con parámetros ?)
sandbox.executemany(
    'INSERT INTO alumnos (nombre, carrera, promedio) VALUES (?, ?, ?)',
    [('Ana', 'LIACD', 92.5), ('Luis', 'LIACD', 85.0), ('Sofía', 'LINF', 78.4)]
)
sandbox.commit()

pd.read_sql_query('SELECT * FROM alumnos', sandbox)

,id,nombre,carrera,promedio
0,1,Ana,LIACD,92.5
1,2,Luis,LIACD,85.0
2,3,Sofía,LINF,78.4


In [ ]:
# ── UPDATE y DELETE ────────────────────────────────────────────────────────

sandbox.execute("UPDATE alumnos SET promedio = 88.0 WHERE nombre = 'Luis'")
sandbox.execute("DELETE FROM alumnos WHERE carrera = 'LINF'")
sandbox.commit()

pd.read_sql_query('SELECT * FROM alumnos', sandbox)

,id,nombre,carrera,promedio
0,1,Ana,LIACD,92.5
1,2,Luis,LIACD,88.0


In [ ]:
# ── Las restricciones (constraints) protegen la calidad ────────────────────

try:
    sandbox.execute("INSERT INTO alumnos (nombre, promedio) VALUES ('Pedro', 150)")
except sqlite3.IntegrityError as e:
    print()
    print(f'Fila rechazada: {e}')
    print('El CHECK impidió guardar un promedio fuera de rango.')
    print()

sandbox.close()


Fila rechazada: CHECK constraint failed: promedio BETWEEN 0 AND 100
El CHECK impidió guardar un promedio fuera de rango.



> **Importante:** un `UPDATE` o `DELETE` **sin `WHERE`** afecta a **todas** las filas de la tabla. Antes de ejecutar uno, corre primero un `SELECT` con el mismo `WHERE` para ver qué filas vas a tocar.

---
## PASO 13 — Ejercicios con tus propios datos

Aplica lo aprendido sobre **tus 3 tablas `raw_<etiqueta>`** de la Actividad 1. Primero revisa sus nombres y columnas con el Paso 2.

| # | Ejercicio | Cláusulas |
|---|---|---|
| 1 | Lista las tablas de tu `.db` y describe las columnas de cada una de tus 3 tablas | `sqlite_master`, `PRAGMA` |
| 2 | Muestra 10 registros de una tabla, solo con 4 columnas que te parezcan importantes y con alias en español | `SELECT`, `AS`, `LIMIT` |
| 3 | Filtra los registros que cumplan **dos** condiciones a la vez | `WHERE`, `AND` |
| 4 | Obtén un **Top 5** según una columna numérica | `ORDER BY ... DESC LIMIT` |
| 5 | Crea una columna con `CASE WHEN` que clasifique los registros en 3 categorías | `CASE` |
| 6 | Resume una columna numérica por una categórica y quédate solo con los grupos que tengan más de N registros | `GROUP BY`, `HAVING` |
| 7 | Cuenta los nulos de 3 columnas y busca duplicados: ¿cumple tu tabla las reglas de calidad de tu miniespecificación? | `CASE`, `GROUP BY`, `HAVING` |
| 8 | **Reto:** si dos de tus tablas comparten una columna (país, fecha, equipo, etc.), únelas con un `JOIN`. Si no comparten ninguna, explica qué clave necesitarías | `JOIN` |

In [ ]:
# ── Ejercicio 1 ────────────────────────────────────────────────────────────

consulta('''
    -- tu consulta aquí
    SELECT name FROM sqlite_master WHERE type = 'table'
''')

,name
0,raw_df1


In [ ]:
# ── Ejercicio 2 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 3 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 4 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 5 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 6 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 7 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

In [ ]:
# ── Ejercicio 8 ────────────────────────────────────────────────────────────
# Escribe tu SQL dentro de las comillas y quita el # de la última línea.

mi_sql = '''
    -- tu consulta aquí
'''

# consulta(mi_sql)

---
## Cierre — liberar la conexión

Siempre cerramos la conexión al terminar (último paso del ciclo **connect → operar → close**). Como solo leímos, no hace falta `commit()`. Al cerrar, la tabla temporal `catalogo_plataformas` también desaparece.

In [ ]:
conn.close()

print()
print('Conexión cerrada. base_datos.db no fue modificado.')
print()


Conexión cerrada. base_datos.db no fue modificado.



### Resumen: SQL que usamos hoy

| Necesito… | Uso |
|---|---|
| Ver qué hay en el `.db` | `SELECT name FROM sqlite_master WHERE type='table'` · `PRAGMA table_info(t)` |
| Elegir columnas | `SELECT col1, col2 AS alias FROM t` |
| Filtrar filas | `WHERE` con `=`, `IN`, `BETWEEN`, `LIKE`, `IS NULL` |
| Ordenar / Top n | `ORDER BY col DESC LIMIT n` |
| Calcular / clasificar | `ROUND`, `CAST`, `COALESCE`, `CASE WHEN` |
| Resumir por grupo | `GROUP BY` + `COUNT/SUM/AVG/MIN/MAX` · `HAVING` |
| Pasos intermedios | subconsultas `( ... )` · `WITH nombre AS ( ... )` |
| Unir tablas | `INNER JOIN` / `LEFT JOIN ... ON clave` |
| Ranking sin agrupar | `RANK() OVER (PARTITION BY ... ORDER BY ...)` |
| Valores desde variables | marcador `?` + `params=(...)` |
| Proteger RAW | `sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)` |